In [12]:
##create samples
import numpy as np
import pandas as pd

np.random.seed(42)

N_ROBOTS = 30
N_DAYS = 14
INTERVAL_MINUTES = 5

samples_per_day = 24 * 60 // INTERVAL_MINUTES
samples_per_robot = samples_per_day * N_DAYS

print("Samples per robot:", samples_per_robot)
print("Total samples:", samples_per_robot * N_ROBOTS)

Samples per robot: 4032
Total samples: 120960


In [13]:
## modes
MODES = ["PATROL", "ALERT", "CHARGING", "FAULT"]

mode_probabilities = {
    "PATROL": 0.65,
    "ALERT": 0.15,
    "CHARGING": 0.15,
    "FAULT": 0.05
}

In [14]:
## make the model to sustain for certain period
def generate_mode_sequence(n_samples):
    modes = []

    while len(modes) < n_samples:
        mode = np.random.choice(
            MODES,
            p=[
                mode_probabilities["PATROL"],
                mode_probabilities["ALERT"],
                mode_probabilities["CHARGING"],
                mode_probabilities["FAULT"]
            ]
        )

        if mode == "PATROL":
            duration = np.random.randint(12, 48)   # 1–4 hours

        elif mode == "ALERT":
            duration = np.random.randint(3, 12)    # 15–60 min

        elif mode == "CHARGING":
            duration = np.random.randint(12, 36)   # 1–3 hours

        else:  # FAULT
            duration = np.random.randint(3, 18)    # 15–90 min

        modes.extend([mode] * duration)

    return modes[:n_samples]

In [15]:
## modor generate sensor values
def generate_sensor_values(mode, t, battery_soc):
    
    if mode == "PATROL":
        accel_x = 0.4 * np.sin(2 * np.pi * t / 12) + np.random.normal(0, 0.15)
        accel_y = 0.3 * np.sin(2 * np.pi * t / 10) + np.random.normal(0, 0.15)
        accel_z = 9.81 + np.random.normal(0, 0.12)

        motor_current = (
            2.0
            + 0.4 * np.sin(2 * np.pi * t / 15)
            + np.random.normal(0, 0.15)
        )

        proximity = np.clip(
            3.0 + np.random.normal(0, 1.0),
            0.2,
            8.0
        )

        rssi = np.random.normal(-58, 5)

        battery_soc -= np.random.uniform(0.01, 0.04)

        task_success = np.random.choice(
            [1, 0],
            p=[0.95, 0.05]
        )
        
    elif mode == "ALERT":
        accel_x = (
                0.9 * np.sin(2 * np.pi * t / 7)+ np.random.normal(0, 0.3)
            )

        accel_y = (
            0.7 * np.sin(2 * np.pi * t / 6)
            + np.random.normal(0, 0.3)
        )

        accel_z = 9.81 + np.random.normal(0, 0.3)

        motor_current = (
            3.3
            + 0.8 * np.sin(2 * np.pi * t / 8)
            + np.random.normal(0, 0.3)
        )

        proximity = np.clip(
            1.5 + np.random.normal(0, 0.7),
            0.1,
            6.0
        )

        rssi = np.random.normal(-60, 6)

        battery_soc -= np.random.uniform(0.03, 0.07)

        task_success = np.random.choice(
            [1, 0],
            p=[0.9, 0.1]
        )
        
    elif mode == "CHARGING":

        accel_x = np.random.normal(0, 0.03)
        accel_y = np.random.normal(0, 0.03)
        accel_z = 9.81 + np.random.normal(0, 0.03)

        motor_current = np.random.normal(0.2, 0.05)

        proximity = np.clip(
            np.random.normal(0.4, 0.1),
            0.1,
            1.0
        )

        rssi = np.random.normal(-45, 3)

        battery_soc += np.random.uniform(0.08, 0.2)

        task_success = 1
        
    elif mode == "FAULT":

        accel_x = np.random.normal(0, 0.8)
        accel_y = np.random.normal(0, 0.8)
        accel_z = 9.81 + np.random.normal(0, 0.7)

        motor_current = (
            4.5
            + 1.2 * np.sin(2 * np.pi * t / 3)
            + np.random.normal(0, 0.7)
        )

        proximity = np.clip(
            np.random.normal(2.0, 1.5),
            0.1,
            8.0
        )

        rssi = np.random.normal(-72, 8)

        battery_soc -= np.random.uniform(0.05, 0.12)

        task_success = np.random.choice(
            [1, 0],
            p=[0.3, 0.7]
        )
        
    battery_soc = np.clip(battery_soc, 0, 100)

    return (
        accel_x,
        accel_y,
        accel_z,
        motor_current,
        proximity,
        rssi,
        battery_soc,
        task_success
    )

In [16]:
rows = []

start_time = pd.Timestamp("2026-08-10 00:00:00")

for robot_num in range(1, N_ROBOTS + 1):

    robot_id = f"R{robot_num:02d}"

    timestamps = pd.date_range(
        start=start_time,
        periods=samples_per_robot,
        freq="5min"
    )

    modes = generate_mode_sequence(samples_per_robot)

    battery_soc = np.random.uniform(60, 100)

    for t, (timestamp, mode) in enumerate(zip(timestamps, modes)):

        (
            accel_x,
            accel_y,
            accel_z,
            motor_current,
            proximity,
            rssi,
            battery_soc,
            task_success
        ) = generate_sensor_values(
            mode,
            t,
            battery_soc
        )

        rows.append({
            "timestamp": timestamp,
            "robot_id": robot_id,
            "mode": mode,
            "accel_x": accel_x,
            "accel_y": accel_y,
            "accel_z": accel_z,
            "motor_current": motor_current,
            "proximity": proximity,
            "rssi": rssi,
            "battery_soc": battery_soc,
            "task_success": task_success
        })

In [17]:
df = pd.DataFrame(rows)

df.head()

,timestamp,robot_id,mode,accel_x,accel_y,accel_z,motor_current,proximity,rssi,battery_soc,task_success
0,2026-08-10 00:00:00,R01,PATROL,0.181560,-0.022685,9.765057,1.806318,2.234491,-55.888431,93.100778,0
1,2026-08-10 00:05:00,R01,PATROL,0.003813,-0.122254,9.707094,2.288695,4.781291,-61.387078,93.089032,1
2,2026-08-10 00:10:00,R01,PATROL,0.114086,0.233848,9.832063,2.383943,5.134641,-56.865756,93.065962,1
3,2026-08-10 00:15:00,R01,PATROL,0.412985,0.196385,9.728399,2.340912,2.719345,-61.019880,93.026576,1
4,2026-08-10 00:20:00,R01,PATROL,0.264022,0.340855,10.063573,2.323048,2.675605,-55.376955,93.016117,1


In [18]:
df.shape

(120960, 11)

In [19]:
df["mode"].value_counts()

mode
PATROL      96458
CHARGING    17102
ALERT        5244
FAULT        2156
Name: count, dtype: int64

In [20]:
df["mode"].value_counts(normalize=True)

mode
PATROL      0.797437
CHARGING    0.141386
ALERT       0.043353
FAULT       0.017824
Name: proportion, dtype: float64